# M05-P03 — Enterprise Document Analyzer

En M01 y M04 `Analyze` trabajó con incidencias. Ahora ampliamos la misma superficie para analizar documentos empresariales más generales:

```text
correo / ticket / política
        ↓
salida estructurada y validada
```

El objetivo es distinguir extracción de hechos, entidades y acciones explícitas de las recomendaciones generadas por el modelo.


In [ ]:
import json
from pathlib import Path

import pandas as pd

from enterprise_genai.document_analysis import (
    EnterpriseDocumentAnalysis,
    analyze_enterprise_document,
    enterprise_document_json_schema,
)

SAMPLES_PATH = Path("data/documents/m05_document_samples.json")
samples = json.loads(SAMPLES_PATH.read_text(encoding="utf-8"))

print("Documentos de prueba:", len(samples))
display(pd.DataFrame([{"id": d["id"], "tipo": d["label"]} for d in samples]))


## 1. El contrato de salida

Antes de llamar al modelo inspeccionamos el JSON Schema. La aplicación no espera texto libre: espera una estructura concreta que Pydantic validará.


In [ ]:
schema = enterprise_document_json_schema()
print("Campos obligatorios:")
print(schema["required"])
print("\nTipos de entidad permitidos:")
print(schema["$defs"]["DocumentEntity"]["properties"]["type"])


## 2. Analizar un correo empresarial

El correo contiene fechas, responsables, estado actual y acciones explícitamente solicitadas. Queremos comprobar que el modelo no confunda esos hechos con recomendaciones propias.


In [ ]:
email_doc = samples[0]
print(email_doc["text"])

email_result = analyze_enterprise_document(email_doc["text"])
email_analysis = email_result["analysis"]

print("\n--- SALIDA VALIDADA ---")
print(email_analysis.model_dump_json(indent=2))
print("\nUso:", email_result["usage"])
print("Latencia Bedrock:", email_result["metrics"].get("latencyMs"), "ms")
print("Stop reason:", email_result["stop_reason"])


## 3. Hechos, acciones solicitadas y recomendaciones no son lo mismo

Esta separación evita presentar como hecho algo que el modelo acaba de proponer.


In [ ]:
print("HECHOS:")
for item in email_analysis.key_facts:
    print("-", item)

print("\nACCIONES SOLICITADAS EXPLÍCITAMENTE:")
for item in email_analysis.requested_actions:
    print("-", item)

print("\nRECOMENDACIONES DEL MODELO:")
for item in email_analysis.recommended_actions:
    print("-", item)

print("\nFECHAS / PLAZOS:")
for item in email_analysis.dates_or_deadlines:
    print("-", item)

print("\nINCERTIDUMBRES:")
for item in email_analysis.uncertainties:
    print("-", item)


## 4. El mismo contrato para documentos distintos

Analizamos ahora un ticket y una política. El schema es el mismo, pero cambian el tipo de documento, la categoría, las entidades y las acciones.


In [ ]:
results = [email_result]

for doc in samples[1:]:
    print("Analizando:", doc["label"])
    results.append(analyze_enterprise_document(doc["text"]))

rows = []
for source, result in zip(samples, results):
    a = result["analysis"]
    rows.append({
        "id": source["id"],
        "document_type": a.document_type,
        "category": a.category,
        "entities": len(a.entities),
        "facts": len(a.key_facts),
        "requested_actions": len(a.requested_actions),
        "recommended_actions": len(a.recommended_actions),
        "dates_deadlines": len(a.dates_or_deadlines),
        "uncertainties": len(a.uncertainties),
        "human_review": a.human_review_required,
        "input_tokens": result["usage"].get("inputTokens"),
        "output_tokens": result["usage"].get("outputTokens"),
        "latency_ms": result["metrics"].get("latencyMs"),
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)


## 5. Entidades con evidencia

No basta con extraer un nombre. Cada entidad incluye una evidencia textual breve que permite revisar por qué el modelo la identificó.


In [ ]:
for source, result in zip(samples, results):
    print("\n", source["id"], "—", result["analysis"].title)
    entity_rows = [entity.model_dump() for entity in result["analysis"].entities]
    display(pd.DataFrame(entity_rows))


## 6. Abrir la caja: validación estructurada

La respuesta cruda de Bedrock es JSON. Pydantic la convierte en `EnterpriseDocumentAnalysis`. Si faltase un campo obligatorio o apareciese un tipo no permitido, la validación fallaría en lugar de ocultar el problema.


In [ ]:
print("Clase validada:", type(email_analysis).__name__)
print("¿Es EnterpriseDocumentAnalysis?:", isinstance(email_analysis, EnterpriseDocumentAnalysis))
print("\nPrimeros 800 caracteres de la respuesta cruda:")
print(email_result["raw_text"][:800])


## 7. Puente a la aplicación

Ejecuta:

```bash
streamlit run app/Home.py
```

En `Analyze`, selecciona **Documento empresarial**. La misma aplicación permite ahora trabajar con:

```text
Incidencia
→ análisis operacional / multimodal

Documento empresarial
→ resumen / clasificación / entidades / acciones
```

### MODIFICA

Si queda tiempo, cambia una fecha, elimina una acción explícita o añade una ambigüedad al documento y observa qué cambia en `dates_or_deadlines`, `requested_actions` o `uncertainties`.
